In [0]:
%pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt

In [0]:
# Imports
import logging
import requests
import time
import base64
from bs4 import BeautifulSoup
from pypdf import PdfReader
from io import BytesIO

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Catalog / Schema / Volume
catalog = "workspace"
schema = "ai_project"
volume = "raw_data"

# Base Volume Path
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"

# Docs Paths
docs_landing_path = vol_path + "docs/raw"

# Doc Sources Table
doc_sources_table = f"{catalog}.{schema}.doc_sources"

# GitHub API Token 
github_token = dbutils.secrets.get(scope="ai-project-secrets", key="github-token")

# %pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt
# dbutils.library.restartPython()

In [0]:
%run ./utils/logging_utils

In [0]:
#  Clean docs from doc_sources
def clean_html(html_content):
    """
    Strips HTML tags and cleans raw page content into plain text.

    Args:
        html_content (str): Raw HTML string from requests
    
    Returns:
        str: Cleaned plain text
    """
    soup = BeautifulSoup(html_content, "html.parser")

    # Remove navigation, headers, footers, scripts, styles
    for element in soup(["nav", "header", "footer", "script", "style", "aside"]):
        element.decompose()

    # Extract plain text
    text = soup.get_text(separator="\n")

    # Clean up excessive whitespace and blank lines
    lines = [line.strip() for line in text.splitlines()]
    cleaned = "\n".join(line for line in lines if line)

    return cleaned


In [0]:
def fetch_with_scrape(url, file_path):
    """Fetches a URL using requests + BeautifulSoup and saves cleaned text to file_path."""
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    response.encoding = "utf-8"

    cleaned_text = clean_html(response.text)

    if not cleaned_text.strip():
        raise ValueError("Cleaned content is empty — page may be JS-rendered.")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

    return round(len(cleaned_text.encode("utf-8")) / 1024, 2)

In [0]:
def fetch_with_api(url, file_path):
    """Fetches file content from GitHub REST API and saves as .txt to file_path."""
    headers = {
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github+json"
    }

    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    data = response.json()
    content_b64 = data.get("content", "")
    encoding = data.get("encoding", "")

    if encoding != "base64":
        raise ValueError(f"Unexpected encoding from GitHub API: {encoding}")

    raw_bytes = base64.b64decode(content_b64)

    # Handle PDF vs plain text
    if url.endswith(".pdf"):
        reader = PdfReader(BytesIO(raw_bytes))
        text = "\n".join(page.extract_text() for page in reader.pages if page.extract_text())
        if not text.strip():
            raise ValueError("PDF content is empty after extraction.")
    else:
        text = raw_bytes.decode("utf-8")
        if not text.strip():
            raise ValueError("File content is empty.")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

    return round(len(text.encode("utf-8")) / 1024, 2)

In [0]:
def fetch_docs():
    """
    Queries doc_sources for active URLs, routes each to the appropriate
    fetch method based on fetch_method column, and saves as .txt to docs/raw.
    """
    sources_df = spark.sql(f"""
        SELECT doc_id, url, title, topic, fetch_method
        FROM {doc_sources_table}
        WHERE active = true
        AND (last_fetched IS NULL OR last_fetched < current_timestamp() - INTERVAL 30 DAYS)
    """)

    sources = sources_df.collect()

    if not sources:
        logger.warning("⚠️ No active sources found in doc_sources.")
        return

    logger.info(f"✅ Found {len(sources)} active sources to fetch.")

    for row in sources:
        doc_id = row["doc_id"]
        url = row["url"]
        title = row["title"]
        topic = row["topic"]
        fetch_method = row["fetch_method"]

        file_name = f"{topic}_{title}.txt".replace(" ", "_").replace("/", "-")
        file_path = f"{docs_landing_path}/{file_name}"

        try:
            logger.info(f"🌐 Fetching [{fetch_method}]: {url}")

            if fetch_method == "api":
                file_size_kb = fetch_with_api(url, file_path)
            elif fetch_method == "scrape":
                file_size_kb = fetch_with_scrape(url, file_path)
            else:
                raise ValueError(f"Unknown fetch_method: {fetch_method}")

            logger.info(f"✅ Saved: {file_name} ({file_size_kb:.2f} KB)")

            spark.sql(f"""
                UPDATE {doc_sources_table}
                SET last_fetched = current_timestamp()
                WHERE doc_id = {doc_id}
            """)

            write_ingestion_log(
                doc_id=doc_id,
                url=url,
                status="SUCCESS",
                file_name=file_name,
                file_size_kb=file_size_kb,
                fetch_method=fetch_method
            )

        except Exception as e:
            logger.error(f"❌ Failed to fetch {url}: {e}")

            write_ingestion_log(
                doc_id=doc_id,
                url=url,
                status="FAILED",
                error_message=str(e),
                fetch_method=fetch_method
            )

    logger.info("🏁 Doc fetching complete.")

In [0]:
# Run
fetch_docs()

In [0]:
dbutils.fs.ls("/Volumes/workspace/ai_project/raw_data/docs/raw/")